# Fine-Tuning SraVaani for Low-Resource Indian Languages: A Wancho Case Study

*Adapting a 63-language Conformer speech recognizer to a language it barely knows, on a single 15GB GPU.*

This notebook walks through fine-tuning [SraVaani](https://huggingface.co/ARTPARK-IISc/SraVaani-1.0) — a Hybrid RNN-T + CTC Conformer ASR model pretrained on 63 Indian languages — on **Wancho**, a low-resource Tibeto-Burman language spoken in Arunachal Pradesh, India.

**TL;DR**
- Base model: [SraVaani](https://huggingface.co/ARTPARK-IISc/SraVaani-1.0) Hybrid RNN-T + CTC BPE Conformer, 63 Indian languages, from ARTPARK-IISc
- Target language: Wancho (not well represented in the pretraining mix)
- Data: [`ARTPARK-IISc/Vaani-transcription-part`](https://huggingface.co/datasets/ARTPARK-IISc/Vaani-transcription-part) on the Hugging Face Hub, Wancho config
- Hardware: 1x NVIDIA T4 (15GB VRAM)

## 1. Before You Begin

You'll need:

- A CUDA-capable GPU. We used a **Tesla T4 (15GB VRAM)** — the smallest end of what's workable for RNN-T fine-tuning.
- At least a few GB of free disk (checkpoint is ~1.8GB, the Wancho dataset is ~1.4GB).
- A Hugging Face account + token with access to `ARTPARK-IISc/Vaani-transcription-part`.

## 2. Environment Setup

Create a virtual environment and install NeMo with ASR support:

```bash
python3 -m venv venv
source venv/bin/activate
pip install "nemo_toolkit[asr,cu13]" pytorch-lightning omegaconf
```

> **Gotcha:** `pip install nemo_toolkit[asr]` alone is not enough. NeMo's RNN-T/TDT loss needs the `numba-cuda` GPU kernel backend, which only comes via the `cu12`/`cu13` extra (match your CUDA driver version from `nvidia-smi`). Without it, everything works until the first training step, which fails deep in a numba kernel compile with an unhelpful `TypeError: Signature mismatch` error. Fix: `pip install "nemo_toolkit[asr,cu13]"`.

In [ ]:
import torch
print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())

import nemo
print("nemo_toolkit:", nemo.__version__)

# Confirms the RNN-T CUDA kernel backend is actually present —
# if this import fails, you hit the cu12/cu13 gotcha above.
import numba_cuda
print("numba-cuda: OK")

## 3. Download the Pretrained Checkpoint

The `.nemo` checkpoint (~1.8GB) is hosted on Google Drive rather than the Hub, so `gdown` handles the large-file confirmation flow that a plain `wget`/`curl` chokes on:

```bash
pip install gdown
gdown "1v5VaYibAaDSFuWvROsPbCzeG3iM6VbxY" -O SraVaani-nemo-checkpoint.nemo
```

Alternatively, [download it manually from Google Drive](https://drive.google.com/drive/folders/1QdMG0R3oGgOThm7uqYeMnt8zGOSx467x) and place it in the project root. Don't have access? [Request access here](https://script.google.com/macros/s/AKfycbyAC0vGcGSbu55LNKOn_hY3TsPy8CyF1nasLJULWMrzCdksBJG33bUfLbiKee_m_aKH-w/exec) — access will be shared ASAP.

**Always verify the download.** A `.nemo` file is just a tar archive; a truncated or interrupted download still produces a file that *looks* plausible by size but fails the moment NeMo tries to read past the point where the download died:

```python
import tarfile
with tarfile.open("SraVaani-nemo-checkpoint.nemo") as tf:
    print(tf.getnames())  # raises tarfile.ReadError on a truncated file
```

If this raises `ReadError: unexpected end of data`, re-download — don't debug the training script first. We lost time doing exactly that before checking the checkpoint itself.

## 4. Prepare the Dataset

This notebook expects data in NeMo's tarred format: a **tarred audio archive** (`shard_NNNN.tar`) paired with a **JSONL manifest** (`shard_NNNN.json`), one line per clip, audio 16kHz mono 16-bit PCM:

```json
{"audio_filepath": "train_00000.wav", "text": "pha ama gaga aga gih pan tega mai tai", "duration": 5.547}
```

`audio_filepath` is the filename **inside** the tar, not a filesystem path.

Prepare `train`/`validation`/`test` splits in this format for your own use case before continuing — how you get there depends on where your data lives (a Hugging Face dataset, local files, etc.).

## 5. Load the Pretrained Model and Freeze the Encoder

With ~5-9 hours of Wancho audio against a model pretrained on 63 languages, full fine-tuning risks catastrophic forgetting of everything the encoder already knows about general Indian-language phonetics. **Freezing the encoder and only training the decoder + joint network** is the safer starting point — it adapts the parts of the model that are language-specific (the prediction network and output vocabulary mapping) while leaving the acoustic feature extraction untouched.

To fine-tune end-to-end instead, just skip the `model.encoder.freeze()` call in the cell below (or call `model.encoder.unfreeze()` before training if you restored a checkpoint that had it frozen). Worth it once the frozen-encoder pass confirms the pipeline works, or if your target language's phonetics differ enough from the pretraining mix that the frozen encoder is the bottleneck — but expect to need a lower learning rate (the encoder is the most sensitive part of the model to disturb) and more GPU memory, since AdamW now also has to hold optimizer state for the encoder's parameters.

In [ ]:
from nemo.collections.asr.models import EncDecHybridRNNTCTCBPEModel

NEMO_CHECKPOINT = "SraVaani-nemo-checkpoint.nemo"

model = EncDecHybridRNNTCTCBPEModel.restore_from(NEMO_CHECKPOINT)

model.encoder.freeze() #optional: freeze encoder weights to only fine-tune decoder/joint weights
print("Encoder frozen — only decoder/joint weights will be updated.")
# To fine-tune end-to-end instead, simply skip this line.

## 6. Configure Data Loaders and Optimizer

A few choices here are tuned specifically for a small, cheap GPU rather than "what NeMo defaults to".

In [ ]:
from omegaconf import OmegaConf, open_dict

TRAIN_SHARDS, TRAIN_MANIFESTS = "data/train/shard_0000.tar", "data/train/shard_0000.json"
VAL_SHARDS, VAL_MANIFESTS = "data/validation/shard_0000.tar", "data/validation/shard_0000.json"

MAX_EPOCHS = 2
BATCH_SIZE = 4     # kept small — see the memory section below
NUM_WORKERS = 1    # kept small — see the memory section below
ACCUMULATE = 8     # effective batch size = BATCH_SIZE * ACCUMULATE = 32

with open(TRAIN_MANIFESTS, encoding="utf-8") as f:
    n_train = sum(1 for line in f if line.strip())

# batches/epoch (ceil, dataloader granularity) -> optimizer steps/epoch (// ACCUMULATE) -> total
MAX_STEPS = (-(-n_train // BATCH_SIZE) // ACCUMULATE) * MAX_EPOCHS
print(f"{n_train} train clips -> {MAX_STEPS} total optimizer steps over {MAX_EPOCHS} epochs")

with open_dict(model.cfg):
    model.cfg.train_ds.manifest_filepath = TRAIN_MANIFESTS
    model.cfg.train_ds.tarred_audio_filepaths = TRAIN_SHARDS
    model.cfg.train_ds.is_tarred = True
    model.cfg.train_ds.shuffle_n = 2048
    model.cfg.train_ds.batch_size = BATCH_SIZE
    model.cfg.train_ds.num_workers = NUM_WORKERS
    model.cfg.train_ds.pin_memory = True

    model.cfg.validation_ds.manifest_filepath = VAL_MANIFESTS
    model.cfg.validation_ds.tarred_audio_filepaths = VAL_SHARDS
    model.cfg.validation_ds.is_tarred = True
    model.cfg.validation_ds.batch_size = BATCH_SIZE
    model.cfg.validation_ds.num_workers = NUM_WORKERS

    # Keep LR small — we're nudging a pretrained model, not training from scratch
    model.cfg.optim.lr = 1e-4
    model.cfg.optim.name = "adamw"
    model.cfg.optim.weight_decay = 1e-3
    model.cfg.optim.sched = OmegaConf.create({
        "name": "CosineAnnealing",
        "warmup_steps": 10,     # small fraction of MAX_STEPS for a short fine-tune run
        "max_steps": MAX_STEPS,
        "min_lr": 1e-6,
    })

model.setup_training_data(model.cfg.train_ds)
model.setup_validation_data(model.cfg.validation_ds)
model.setup_optimization(model.cfg.optim)

## 7. Train

A couple of additions beyond a bare `trainer.fit(model)`:

- A small `MetricsLogger` callback prints `train_loss` every step and `val_wer` at every validation pass — useful when you're watching a run live rather than only reviewing TensorBoard after the fact.
- `exp_manager` handles checkpointing (top-3 by `val_wer`) and TensorBoard logging.

In [ ]:
import lightning.pytorch as pl
from nemo.utils.exp_manager import exp_manager


class MetricsLogger(pl.Callback):
    """Prints train loss every step and val WER at every eval step."""

    def on_train_batch_end(self, trainer, pl_module, outputs, batch, batch_idx):
        loss = trainer.callback_metrics.get("train_loss")
        if loss is not None:
            print(f"[step {trainer.global_step}] train_loss={loss.item():.4f}")

    def on_validation_epoch_end(self, trainer, pl_module):
        wer = trainer.callback_metrics.get("val_wer")
        if wer is not None:
            print(f"[step {trainer.global_step}] val_wer={wer.item():.4f}")


trainer = pl.Trainer(
    devices=1,
    accelerator="gpu",
    max_epochs=MAX_EPOCHS,
    accumulate_grad_batches=ACCUMULATE,
    gradient_clip_val=1.0,
    val_check_interval=1.0,
    log_every_n_steps=10,
    logger=False,
    enable_checkpointing=False,
    callbacks=[MetricsLogger()],
)

exp_manager(trainer, {
    "exp_dir": "experiments/",
    "name": "sra-vaani-finetune",
    "create_tensorboard_logger": True,
    "create_checkpoint_callback": True,
    "checkpoint_callback_params": {"monitor": "val_wer", "mode": "min", "save_top_k": 3},
})

trainer.fit(model)
model.save_to("sra-vaani-finetuned.nemo")

## 8. Monitor Training

Everything logged during `trainer.fit()` also lands in the TensorBoard event file under `experiments/`, which we can pull directly with `EventAccumulator` for a quick static plot — handy in a notebook where you don't want to launch a separate TensorBoard server.

In [ ]:
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator
import glob
import matplotlib.pyplot as plt

event_file = glob.glob("experiments/sra-vaani-finetune/*/events.out.tfevents.*")[-1]
ea = EventAccumulator(event_file, size_guidance={"scalars": 0})
ea.Reload()

train_loss = ea.Scalars("train_loss")
steps = [e.step for e in train_loss]
values = [e.value for e in train_loss]

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(steps, values, linewidth=2)
ax.set_xlabel("Optimizer step")
ax.set_ylabel("train_loss")
ax.set_title("SraVaani fine-tune — training loss (Wancho)")
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Evaluate: Original vs. Fine-Tuned

The real test is comparing the **original** and **fine-tuned** checkpoints on the same held-out test set, with the same text cleaning applied to both references and hypotheses before scoring — otherwise you're partly measuring formatting differences, not transcription quality.

In [ ]:
import csv
import shutil
import tempfile

import jiwer

TEST_SHARDS, TEST_MANIFEST = "data/wancho/test/shard_0000.tar", "data/wancho/test/shard_0000.json"
ORIGINAL_CHECKPOINT, FINETUNED_CHECKPOINT = "SraVaani-nemo-checkpoint.nemo", "sra-vaani-finetuned.nemo"
INFER_BATCH_SIZE = 32


def load_test_set(shards_path, manifest_path):
    tmp_dir = tempfile.mkdtemp(prefix="wancho_test_")
    with tarfile.open(shards_path) as tar:
        tar.extractall(tmp_dir)
    audio_filepaths, references = [], []
    with open(manifest_path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                entry = json.loads(line)
                audio_filepaths.append(f"{tmp_dir}/{entry['audio_filepath']}")
                references.append(entry["text"])
    return audio_filepaths, references, tmp_dir


def run_inference(checkpoint_path, audio_filepaths):
    m = EncDecHybridRNNTCTCBPEModel.restore_from(checkpoint_path)
    m.eval()
    if torch.cuda.is_available():
        m = m.cuda()
    hyps = m.transcribe(audio=audio_filepaths, batch_size=INFER_BATCH_SIZE)
    hyps = [h.text if hasattr(h, "text") else h for h in hyps]
    del m
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return hyps


audio_filepaths, references, tmp_dir = load_test_set(TEST_SHARDS, TEST_MANIFEST)
print(f"Loaded {len(audio_filepaths)} test clips")

hyp_original = run_inference(ORIGINAL_CHECKPOINT, audio_filepaths)
hyp_finetuned = run_inference(FINETUNED_CHECKPOINT, audio_filepaths)
shutil.rmtree(tmp_dir, ignore_errors=True)

# References and both sets of hypotheses go through the same clean() from §4
refs_clean = [clean(r) for r in references]
hyp_original_clean = [clean(h) for h in hyp_original]
hyp_finetuned_clean = [clean(h) for h in hyp_finetuned]

wer_original = jiwer.wer(refs_clean, hyp_original_clean)
wer_finetuned = jiwer.wer(refs_clean, hyp_finetuned_clean)

print(f"Original checkpoint  WER: {wer_original:.4f} ({wer_original * 100:.2f}%)")
print(f"Fine-tuned checkpoint WER: {wer_finetuned:.4f} ({wer_finetuned * 100:.2f}%)")

with open("inference_results.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["audio_filepath", "reference", "hyp_original", "hyp_finetuned"])
    for path, ref, ho, hf in zip(audio_filepaths, refs_clean, hyp_original_clean, hyp_finetuned_clean):
        writer.writerow([path.rsplit("/", 1)[-1], ref, ho, hf])

## 10. Results

| Checkpoint | WER on Wancho test set |
|---|---|
| Original SraVaani (zero-shot on Wancho) | 65.26% |
| Fine-tuned (2 epochs, encoder frozen) | 64.22% |

Fine-tuning on the full Wancho train split (2 epochs, encoder frozen) took WER from 65.26% to 64.22% — about a 1-point absolute improvement. Modest, consistent with a short, decoder-only fine-tune: the encoder (unchanged) is still doing all the acoustic heavy lifting on a language it saw little of during pretraining, and 2 epochs is enough to adapt the output vocabulary mapping but not enough to meaningfully reshape the language model the decoder carries. See §11 for what would likely move this further.

## 11. Next Steps

- **Unfreeze the encoder** for full fine-tuning once the decoder-only pass shows the setup works end-to-end — expect to need a lower learning rate and closer memory monitoring (encoder gradients roughly double optimizer state size).
- **Train longer / on more data.** 2 epochs on a few hours of audio is a smoke test, not a converged model; the loss curve in §8 plateauing quickly is expected at this scale.
- **Repeat for another of the 63 pretraining languages** that's similarly underrepresented — the pipeline here (download → clean → tar/manifest → freeze-and-finetune → compare-to-baseline) is language-agnostic; only `LANGUAGE` and the checkpoint need to change.
- **Try it on your own data.** Everything here is language- and dataset-agnostic once you've prepared your data in the format from §4 — swap in your own audio/transcripts for your own use case (a dialect, a domain, a language SraVaani wasn't pretrained on at all) and the rest of the notebook runs unchanged.

---

*This notebook documents a real fine-tuning run. Feel free to adapt it for you own usecase. Questions or clarifications about this notebook? Reach out at agneedh@artpark.in.*

---

